# CMSC 173 &middot; Machine Learning &mdash; Week 11 Lab
## Kernel Methods & Support Vector Machines

Some data can't be split by any straight line. The **kernel trick** is a beautiful escape:
lift the data into a higher dimension where it *becomes* linearly separable, without ever
building that dimension explicitly. You'll see a linear model fail, understand *why* the trick
works, then let an **RBF-kernel SVM** carve the curved boundary &mdash; and meet the margin and its
support vectors.

**How this lab works.** Each part = a short **plain-English explainer**, a **code cell**
you run, a **line-by-line walkthrough** of what it did, and an **Answer here** box. The
code does the maths; we *graph* the results so you can see what is going on.

**NumPy + Matplotlib + scikit-learn.** **Not graded.** About 55 minutes.

---
## Part 0 &middot; Setup + data no line can split

`make_circles` gives one class as an inner blob and the other as a surrounding ring. No straight
line can separate a ring from its centre &mdash; perfect for showing why we need kernels.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from sklearn.datasets import make_circles
from sklearn.svm import SVC

X, y = make_circles(n_samples=300, noise=0.08, factor=0.4, random_state=173)

def show(ax):
    ax.scatter(X[y==0,0], X[y==0,1], marker='o', alpha=0.6, label='class 0 (ring)')
    ax.scatter(X[y==1,0], X[y==1,1], marker='^', alpha=0.6, label='class 1 (centre)')
def regions(ax, model):
    xx, yy = np.meshgrid(np.linspace(-1.6,1.6,300), np.linspace(-1.6,1.6,300))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.25, cmap='coolwarm')
plt.figure(figsize=(5.5,5)); show(plt.gca()); plt.legend(); plt.title('A ring around a centre'); plt.tight_layout(); plt.show()

**Reading the code:** `show` draws the two classes; `regions` will shade whatever a model predicts
across the plane (we reuse both below). Look at the plot &mdash; there is genuinely no straight line
that puts the ring on one side and the centre on the other.

---
## Part 1 &middot; A straight line fails (as it must)

Fit a **linear** SVM and shade its regions. It will do its hopeless best: split the plane with one
line and get about half the ring wrong.

In [ ]:
lin = SVC(kernel='linear').fit(X, y)
print('linear SVM accuracy:', round(lin.score(X, y), 3))
plt.figure(figsize=(5.5,5)); regions(plt.gca(), lin); show(plt.gca())
plt.title('Linear SVM: one line, no hope'); plt.legend(); plt.tight_layout(); plt.show()

**Reading the code:** `SVC(kernel='linear')` is a straight-boundary classifier. Its accuracy hovers
near 50% &mdash; no better than guessing &mdash; and the shaded regions confirm it slices the plane in half,
which can never match a ring. The model isn't broken; the *shape* is wrong.

**Answer here:**

1. Why is ~50% accuracy exactly what you'd expect here, and not a sign the code is buggy?
   &rarr; *your answer*

---
## Part 2 &middot; The trick: add a dimension

Here's the insight. Add a third feature &mdash; distance from the centre, $r = x_1^2 + x_2^2$. In that
lifted space the centre blob sits *low* and the ring sits *high*, so a flat plane can slice between
them. The kernel trick does this lifting *implicitly*; let's do it explicitly once to believe it.

In [ ]:
r = (X**2).sum(axis=1)                                # the new 3rd feature: squared distance
fig = plt.figure(figsize=(6,5)); ax = fig.add_subplot(111, projection='3d')
ax.scatter(X[y==0,0], X[y==0,1], r[y==0], marker='o', alpha=0.5, label='ring')
ax.scatter(X[y==1,0], X[y==1,1], r[y==1], marker='^', alpha=0.5, label='centre')
ax.set_xlabel('x1'); ax.set_ylabel('x2'); ax.set_zlabel('r = x1^2 + x2^2')
ax.set_title('Lifted to 3-D: now a flat plane can split them'); ax.legend(); plt.tight_layout(); plt.show()

**Reading the code:** we compute `r = x1^2 + x2^2` and plot the data in 3-D with `r` as height. The
centre points (small `r`) sink to the bottom; the ring (large `r`) floats up. Now a horizontal
*plane* separates them cleanly. A kernel is just a way to get this separation **without** actually
computing the extra dimensions &mdash; it works with distances between points instead.

**Answer here:**

1. In the 3-D plot, roughly where would you slide a flat plane to separate the classes? What does
   that plane look like back in the original 2-D picture?
   &rarr; *your answer*

---
## Part 3 &middot; RBF-kernel SVM: the curved boundary

The **RBF kernel** does that lifting for you and finds the boundary in the original space &mdash; which
comes out *curved*. Same `SVC`, one word changed.

In [ ]:
rbf = SVC(kernel='rbf', gamma=1.0).fit(X, y)
print('RBF SVM accuracy:', round(rbf.score(X, y), 3))

fig, ax = plt.subplots(1, 2, figsize=(11,5))
regions(ax[0], lin); show(ax[0]); ax[0].set_title('linear kernel (fails)')
regions(ax[1], rbf); show(ax[1]); ax[1].set_title('RBF kernel (a circle!)')
ax[0].legend(); plt.tight_layout(); plt.show()

**Reading the code:** switching `kernel='linear'` to `kernel='rbf'` is the only change, and accuracy
jumps to near 100%. The right-hand regions show a roughly **circular** boundary &mdash; the straight
plane from the 3-D lift, seen back in 2-D. Same algorithm, a smarter notion of 'distance'.

**Answer here:**

1. The only thing we changed was the kernel. In one sentence, what did switching to RBF let the SVM
   do that the linear one couldn't?
   &rarr; *your answer*

---
## Part 4 &middot; Margin & support vectors

An SVM doesn't just find *a* boundary &mdash; it finds the one with the widest **margin** (buffer) to the
nearest points. Those nearest points are the **support vectors**; they alone define the boundary.

In [ ]:
print('number of support vectors:', rbf.n_support_, '(one count per class)')
plt.figure(figsize=(5.5,5)); regions(plt.gca(), rbf); show(plt.gca())
sv = rbf.support_vectors_
plt.scatter(sv[:,0], sv[:,1], s=90, facecolors='none', edgecolors='k', label='support vectors')
plt.title('Circled points are the support vectors'); plt.legend(); plt.tight_layout(); plt.show()

**Reading the code:** `rbf.support_vectors_` are the points on the edge of the margin. We circle
them: notice they sit right along the boundary between ring and centre. Delete a point far from the
boundary and nothing changes; delete a support vector and the boundary moves. That's why SVMs are
memory-efficient &mdash; they only *need* the support vectors.

**Answer here:**

1. Try `gamma=20` in Part 3 instead of `1.0` and re-run. Does the boundary get more wiggly? Which
   failure (over/underfitting) is a huge gamma courting?
   &rarr; *your answer*

---
## Where you actually are

Set the pace honestly. Replace each `-` with: **solid** / **rusty** / **never really got it**.

| | You |
|---|---|
| Why a linear model fails on the ring | - |
| The kernel-trick idea (lift a dimension) | - |
| What the RBF kernel buys you | - |
| Margin & support vectors | - |
| The effect of gamma | - |

**Which part took longest, and where did you get stuck?**
&rarr; *your answer*

**In one plain sentence: what does a kernel let a linear classifier do?**
&rarr; *your answer*

---
## Stretch &mdash; optional

Required part is done; nothing below is graded.

### Stretch &middot; Polynomial kernel

Fit `SVC(kernel='poly', degree=2)` on the same data and print its accuracy. Does a degree-2
polynomial kernel also crack the ring? Fill it in.

In [ ]:
# your code here: SVC(kernel='poly', degree=2).fit(X, y).score(X, y)


---
## Submitting

Run the cell below. It uploads this notebook straight from Colab &mdash; nothing to download.

You need a **submit token**: open
[https://portal.latarak.com/student/submit-token](https://portal.latarak.com/student/submit-token),
sign in, press the button, then paste it when the cell asks. The cell hides what you type.

In [ ]:
# --- Submit this notebook ------------------------------------------------------
# Colab only. Anywhere else, use the manual route described below this cell.
import getpass, json, urllib.request, urllib.error

PORTAL, COURSE, WEEK = "https://portal.latarak.com", "cmsc173", 11

try:
    from google.colab import _message
except ImportError:
    raise SystemExit(
        "Not running in Colab. Download this notebook "
        "(File > Download > Download .ipynb) and upload it at "
        "https://portal.latarak.com/course/cmsc173/lab/11/submit"
    )

nb = _message.blocking_request("get_ipynb", timeout_sec=90)["ipynb"]
token = getpass.getpass("Submit token (hidden as you type): ").strip()

req = urllib.request.Request(
    PORTAL + "/api/labs/" + COURSE + "/submit-notebook",
    data=json.dumps({"week": WEEK, "notebook": nb}).encode(),
    headers={"Content-Type": "application/json", "Authorization": "Bearer " + token},
    method="POST",
)
try:
    with urllib.request.urlopen(req, timeout=120) as r:
        out = json.load(r)
    print("Submitted", out["course"], "week", out["week"], "for", out["student"])
    print(out["cells"], "cells,", out["executed"], "executed")
    print(out["message"])
except urllib.error.HTTPError as e:
    print("Not submitted:", json.loads(e.read()).get("error", e.reason))

Prefer to do it by hand? **File &rarr; Download &rarr; Download .ipynb**, then go to the
[Week 11 submission page](https://portal.latarak.com/course/cmsc173/lab/11/submit) and upload it.

Blank cells are fine and guesses are fine. Don't polish this until it hides what you knew.